# **1. 손글씨 도형**
- 손으로 그린 원, 삼각형, X 이미지를 CNN으로 분류
- 데이터 구성: 학습 데이터 240장, 테스트 60장, 3개의 클래스가 균등하게 구성

In [33]:
# 블로그에는 코랩으로 돌리는 거
import torch
import torch.nn as nn
import torch.optim as optim
# import torchvision
# import torchvision.transforms as transforms
# from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

from pathlib import Path
import random
import numpy as np
import zipfile
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

In [34]:
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [35]:
extract_dir = Path("./data")
if not extract_dir.exists():
    extract_dir.mkdir(parents=True, exist_ok = True) # exist_ok=True는 있어도 에러 안나고 그냥 통과


In [36]:
with zipfile.ZipFile("./data/shape.zip", "r") as zip_ref:
    zip_ref.extractall("./data") # 현재 폴더에 압축 해제하고 싶으면 ()로 끝내도 됨

In [37]:
shape_root = extract_dir / 'shape'
train_dir = shape_root / 'train'
test_dir = shape_root / 'test'
print("학습 경로: ", train_dir)
print("테스트 경로: ", test_dir)

학습 경로:  data\shape\train
테스트 경로:  data\shape\test


# **2. ImageFolder**
하위 폴더 이름을 클래스 이름으로 사용하며, 알파벳 순서대로 클래스 번호를 지정함

즉, 폴더명을 가지고 라벨을 정하는 거임

In [40]:
# cir(0), tri(1), x(2)
raw_train = datasets.ImageFolder(train_dir)
raw_test = datasets.ImageFolder(test_dir)

print('클래스: ', raw_train.classes)
print('클래스 번호: ', raw_train.class_to_idx)
print('학습 데이터 이미지 수: ', len(raw_train))
print('테스트 데이터 이미지 수: ', len(raw_test))

클래스:  ['cir', 'tri', 'x']
클래스 번호:  {'cir': 0, 'tri': 1, 'x': 2}
학습 데이터 이미지 수:  240
테스트 데이터 이미지 수:  60


# **3. 전처리와 데이터 증강**
- 모든 이미지를 28x28 1채널 흑백으로 통일함 (지금 흑백이 아니라 컬러로 되어 있음)
- 흰 배경에 검은 선으로 그려진 이미지를 반전하여 선 부분이 큰 값을 갖게 함
- 넘파이로 가져오기 때문에 픽셀 값을 텐서로 바꾼 뒤, 평균 0.5, 표준편차 0.5로 정규화
- 일반화 성능을 높이기 위해 학습 데이터에만 회전/이동/크기 등을 변화

In [45]:
train_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels = 1),
    transforms.RandomInvert(p=1.0), # 1.0이 모든 이미지가 반전됨 0.5면 50프로만 반전
    transforms.RandomAffine(degrees=12, translate=(0.08, 0.08), scale = (0.090, 1.10)),
    # ToTensor(): 데이터를 텐서로 바꾸기도 하지만 스케일도 같이 바뀜 그래서 0 ~ 1 사이로 정규화 됨 0 ~ 255가 아님
    transforms.ToTensor(),
    # 중앙값을 0, 범위를 -1~1로 조절
    transforms.Normalize(mean = (0.5,), std=(0.5,)) # 평균이 0.5 표준편차 0.5
])

eval_transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels = 1),
    transforms.RandomInvert(p=1.0), # 1.0이 모든 이미지가 반전됨 0.5면 50프로만 반전
    transforms.ToTensor(),
    transforms.Normalize(mean = (0.5,), std=(0.5,))
])